# Control Variates in QMCPy

This notebook mirrors the QMCPy `control_variates.ipynb` demo while using Julia's `QuasiMC.jl` APIs. It demonstrates current support for linear control variates in both Monte Carlo and quasi-Monte Carlo stopping criteria.

Parity note: this Julia notebook keeps the same three control-variate problems, seeds, tolerances, and stopping-criterion families. The Asian option uses an explicit trapezoidal payoff over `GeometricBrownianMotion` because QuasiMC.jl's current public `FinancialOption` uses right-endpoint averaging.

Original QMCPy demo: [`QMCPy/demos/control_variates.ipynb`](../../QMCPy/demos/control_variates.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QuasiMC.jl/blob/develop/demos/control_variates.ipynb)


## Setup

In [1]:
using QuasiMC
using Printf
using Statistics

sample_count(result) = haskey(result.data, :n_total) ? result.data[:n_total] : result.data[:n]

function compare(problem, discrete_distrib, stopping_crit; abs_tol)
    g, cvs, cvmus = problem(discrete_distrib)
    sc = stopping_crit(g; abs_tol = abs_tol)
    result = integrate(sc)
    sc_cv = stopping_crit(
        g;
        abs_tol = abs_tol,
        control_variates = cvs,
        control_variate_means = cvmus,
    )
    result_cv = integrate(sc_cv)
    @assert isfinite(result.solution) && isfinite(result_cv.solution)
    @assert sample_count(result) > 0 && sample_count(result_cv) > 0
    name = nameof(typeof(sc))
    @printf("Stopping Criterion: %-15s absolute tolerance: %.1e\n", String(name), abs_tol)
    @printf("  Without CV: solution %-10.5f samples %.1e\n", result.solution, float(sample_count(result)))
    @printf("  With CV:    solution %-10.5f samples %.1e\n", result_cv.solution, float(sample_count(result_cv)))
    @printf("  Sample ratio with CV: %.1f%%\n", 100 * sample_count(result_cv) / sample_count(result))
    beta = result_cv.data[:control_variate_beta]
    @printf("  Control-variate beta: [%s]\n", join([@sprintf("%.4g", b) for b in beta], ", "))
    println()
    return result, result_cv
end

compare (generic function with 1 method)

## Problem 1: Polynomial Function

We will integrate

$$g(t) = 10t_1 - 5t_2^2 + 2t_3^3$$

with true measure $\mathcal{U}[0,2]^3$ and control variates

$$\hat{g}_1(t) = t_1$$

and

$$\hat{g}_2(t) = t_2^2$$

using the same true measure.

In [2]:
function poly_problem(dd)
    tm = Uniform(dd; lower_bound = 0.0, upper_bound = 2.0)
    g = CustomFun(tm, x -> 10 .* x[:, 1] .- 5 .* x[:, 2].^2 .+ x[:, 3].^3)
    cv1 = CustomFun(tm, x -> x[:, 1])
    cv2 = CustomFun(tm, x -> x[:, 2].^2)
    return g, [cv1, cv2], [1.0, 4 / 3]
end

compare(poly_problem, IIDStdUniform(3; seed = 7), CubMCCLT; abs_tol = 1e-2)
compare(
    poly_problem,
    DigitalNetB2(3; seed = 7, randomize = "LMS_DS", graycode = false),
    CubQMCNetG;
    abs_tol = 1e-8,
)


Stopping Criterion: CubMCCLT        absolute tolerance: 1.0e-02
  Without CV: solution 5.33820    samples 7.1e+06


  With CV:    solution 5.33303    samples 4.9e+05
  Sample ratio with CV: 7.0%
  Control-variate beta: [9.943, -5.032]



Stopping Criterion: CubQMCNetG      absolute tolerance: 1.0e-08
  Without CV: solution 5.33334    samples 1.0e+06


  With CV:    solution 5.33334    samples 5.2e+05
  Sample ratio with CV: 50.0%
  Control-variate beta: [9.995, -5]



(QMCResult(solution=5.333339e+00, n_total=1048576, error_bound=5.82e-09), QMCResult(solution=5.333339e+00, n_total=524288, error_bound=3.58e-09))

## Problem 2: Keister Function

This problem integrates the Keister function while using control variates

$$g_1(x) = \sin(\pi x)$$

and

$$g_2(x) = -3(x - 1/2)^2 + 1.$$

As in the QMCPy demo, this Julia version uses the one-dimensional case for readability, but the control-variate API is compatible with higher dimensions.

In [3]:
function keister_problem(dd)
    main = Keister(Gaussian(dd; covariance = 0.5))
    tm = Uniform(dd)
    cv1 = CustomFun(tm, x -> sin.(pi .* x[:, 1]))
    cv2 = CustomFun(tm, x -> -3 .* (x[:, 1] .- 0.5).^2 .+ 1.0)
    return main, [cv1, cv2], [2 / pi, 3 / 4]
end

compare(keister_problem, IIDStdUniform(1; seed = 7), CubMCCLT; abs_tol = 5e-4)
compare(
    keister_problem,
    DigitalNetB2(1; seed = 7, randomize = "LMS_DS", graycode = false),
    CubQMCNetG;
    abs_tol = 1e-7,
)


Stopping Criterion: CubMCCLT        absolute tolerance: 5.0e-04
  Without CV: solution 1.38015    samples 9.4e+06


  With CV:    solution 1.38025    samples 4.5e+05
  Sample ratio with CV: 4.8%
  Control-variate beta: [-9.034, 14.47]

Stopping Criterion: CubQMCNetG      absolute tolerance: 1.0e-07
  Without CV: solution 1.38039    samples 2.1e+06


  With CV:    solution 1.38039    samples 1.0e+06
  Sample ratio with CV: 50.0%
  Control-variate beta: [-40.11, 57.29]



(QMCResult(solution=1.380388e+00, n_total=2097152, error_bound=5.93e-08), QMCResult(solution=1.380388e+00, n_total=1048576, error_bound=7.55e-08))

## Problem 3: Option Pricing

We use a European call option as a control variate for pricing an arithmetic Asian call option, following the same structure as the QMCPy demo.

In [4]:
call_put = :call
start_price = 100.0
strike_price = 125.0
volatility = 0.75
interest_rate = 0.01
t_final = 1.0
dimension = 4

function option_problem(dd)
    tm = GeometricBrownianMotion(
        dd;
        t_final = t_final,
        initial_value = start_price,
        drift = interest_rate,
        diffusion = volatility^2,
    )
    european_cv = FinancialOption(
        tm;
        option_type = :european,
        call_put = call_put,
        volatility = volatility,
        start_price = start_price,
        strike_price = strike_price,
        interest_rate = interest_rate,
    )
    asian_call = CustomFun(tm, function (paths)
        d = size(paths, 2)
        path_sum = vec(sum(@view(paths[:, 1:(d - 1)]); dims = 2))
        average_price = (start_price / 2 .+ path_sum .+ paths[:, d] / 2) ./ d
        return exp(-interest_rate * t_final) .* max.(average_price .- strike_price, 0.0)
    end)
    return asian_call, european_cv, get_exact_value(european_cv)
end

compare(option_problem, IIDStdUniform(dimension; seed = 7), CubMCCLT; abs_tol = 5e-2)
compare(
    option_problem,
    DigitalNetB2(dimension; seed = 7, randomize = "LMS_DS", graycode = false),
    CubQMCNetG;
    abs_tol = 1e-3,
)


Stopping Criterion: CubMCCLT        absolute tolerance: 5.0e-02
  Without CV: solution 9.55822    samples 3.0e+06


  With CV:    solution 9.53139    samples 9.9e+05
  Sample ratio with CV: 32.5%
  Control-variate beta: [0.3353]

Stopping Criterion: CubQMCNetG      absolute tolerance: 1.0e-03
  Without CV: solution 9.55223    samples 5.2e+05


  With CV:    solution 9.55222    samples 5.2e+05
  Sample ratio with CV: 100.0%
  Control-variate beta: [-0.03232]



(QMCResult(solution=9.552228e+00, n_total=524288, error_bound=7.73e-04), QMCResult(solution=9.552219e+00, n_total=524288, error_bound=7.50e-04))